# OpenMP Parallel Consistency

Fresh Python processes execute the same multi-shot forward and backward case
with one, two, and four OpenMP threads. This isolates native OpenMP runtime
initialization and checks thread-count invariance of traces and gradients.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "tests"]
candidates.extend(parent / "tests" for parent in cwd.parents)
NOTEBOOK_DIR = next(
    (path for path in candidates if (path / "verification_utils.py").is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("verification_utils.py was not found from the current directory.")
notebook_path = str(NOTEBOOK_DIR)
if notebook_path not in sys.path:
    sys.path.insert(0, notebook_path)

import verification_utils as vu

REPO_ROOT = vu.configure_local_import()
for module_name in tuple(sys.modules):
    if module_name == "DeepGPR" or module_name.startswith("DeepGPR."):
        del sys.modules[module_name]
import DeepGPR

LOADED_PACKAGE = vu.assert_local_deepgpr(DeepGPR, REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"DeepGPR package: {LOADED_PACKAGE}")


In [ ]:
import os
import subprocess
import sys
import tempfile
import torch

DEVICE = torch.device("cpu")
CHECKS = []
METADATA = vu.runtime_metadata(DeepGPR, DEVICE)
worker = NOTEBOOK_DIR / "openmp_worker.py"
if not worker.is_file():
    raise FileNotFoundError(worker)


In [ ]:
thread_results = {}
with tempfile.TemporaryDirectory(prefix="deepgpr_openmp_") as temporary_directory:
    temporary_path = Path(temporary_directory)
    for thread_count in (1, 2, 4):
        output_path = temporary_path / f"threads_{thread_count}.pt"
        environment = os.environ.copy()
        environment["OMP_NUM_THREADS"] = str(thread_count)
        environment["PYTHONPATH"] = str(REPO_ROOT / "src")
        completed = subprocess.run(
            [sys.executable, str(worker), str(output_path)],
            cwd=NOTEBOOK_DIR,
            env=environment,
            text=True,
            capture_output=True,
            check=False,
        )
        vu.record_check(
            CHECKS,
            f"OpenMP worker exits successfully with {thread_count} thread(s)",
            completed.returncode == 0 and output_path.is_file(),
            returncode=completed.returncode,
            stdout=completed.stdout,
            stderr=completed.stderr,
        )
        thread_results[thread_count] = torch.load(
            output_path, map_location="cpu", weights_only=True
        )


In [ ]:
reference = thread_results[1]
comparison_rows = []
for thread_count in (2, 4):
    candidate = thread_results[thread_count]
    row = {
        "threads": thread_count,
        "requested_threads": candidate["requested_omp_num_threads"],
        "elapsed_seconds": candidate["elapsed_seconds"],
        "receiver_relative_l2": vu.relative_l2(
            candidate["receiver"], reference["receiver"]
        ),
        "er_gradient_relative_l2": vu.relative_l2(
            candidate["grad_er"], reference["grad_er"]
        ),
        "se_gradient_relative_l2": vu.relative_l2(
            candidate["grad_se"], reference["grad_se"]
        ),
        "er_gradient_cosine": vu.cosine_similarity(
            candidate["grad_er"], reference["grad_er"]
        ),
        "se_gradient_cosine": vu.cosine_similarity(
            candidate["grad_se"], reference["grad_se"]
        ),
    }
    comparison_rows.append(row)
    vu.record_check(
        CHECKS,
        f"OpenMP numerical consistency for {thread_count} threads",
        row["requested_threads"] == thread_count
        and row["receiver_relative_l2"] < 1.0e-7
        and max(
            row["er_gradient_relative_l2"],
            row["se_gradient_relative_l2"],
        ) < 2.0e-6
        and min(
            row["er_gradient_cosine"],
            row["se_gradient_cosine"],
        ) > 0.999999,
        **row,
        receiver_tolerance=1.0e-7,
        gradient_tolerance=2.0e-6,
        cosine_tolerance=0.999999,
    )


In [ ]:
vu.save_report(
    "08_openmp_parallelism",
    CHECKS,
    METADATA,
    extra={
        "reference_elapsed_seconds": reference["elapsed_seconds"],
        "comparison_rows": comparison_rows,
    },
)
print(f"Completed {len(CHECKS)} required checks.")
